## Multiprocessing and Synchronization 

#### CPU multiprocessing concepts with GPU multiprocessing

Yes. You can create multiple processes to `execute ML workloads` using CUDA GPUs and synchronize their execution. First, understand one important distinction:

**CUDA multiprocessing is not the same as CUDA multithreading.**

* **Multiprocessing:** Multiple CPU processes execute workloads, potentially using the same or different GPUs.
* **Multithreading:** Multiple CPU threads execute within a process.
* **CUDA streams:** Multiple GPU execution queues allow operations to overlap.
* **CUDA threads:** Thousands of GPU threads execute inside CUDA thread blocks.

## CUDA multiprocessing architecture

Consider a system with one NVIDIA GPU. You create three CPU processes, and each process submits GPU workloads.

```mermaid
flowchart TD
    Main["<b>CPU – Main Process</b><br/>Process Manager"]
    P1["<b>Process 1</b><br/>CPU worker"]
    P2["<b>Process 2</b><br/>CPU worker"]
    P3["<b>Process 3</b><br/>CPU worker"]
    GPU["<b>NVIDIA GPU</b><br/>CUDA contexts, streams and GPU kernels"]
    W1["GPU workload 1"]
    W2["GPU workload 2"]
    W3["GPU workload 3"]

    Main --> P1
    Main --> P2
    Main --> P3
    P2 -- "CUDA runtime / GPU submission" --> GPU
    GPU --> W1
    GPU --> W2
    GPU --> W3
```

Each process can submit CUDA operations. However, GPU execution may be serialized or overlapped depending on GPU resources, stream dependencies, and workload characteristics.


## Create multiprocessing using PyTorch

We need to know some API

- `mp.set_start_method(method, force=False)`:
    - method three
      1. spawn -> `mp.set_start_method("spawn")`
            - Creates a completely new Python interpreter.
            - Process flow
                ```table
                Main Process
                      │
                      ▼
                New Python Interpreter
                      │
                Import script again
                      │
                Execute target function
                ```
            - Internally
                  ```table
                  GPU initialized
                        ↓
                  fork()
                        ↓
                  Child inherits CUDA state
                  ```
            - Each process has:
                  - Separate memory
                  - Separate variables
                  - Separate CUDA context
      - Why PyTorch CUDA Uses Spawn?
            - CUDA cannot safely be copied using fork.
            - Bad:
                  ```text
                  GPU initialized
                       ↓
                    fork()
                       ↓
                  Child inherits CUDA state
                  ```
            - Can cause:                  
                - CUDA initialization errors
                - Deadlocks
                - Memory corruption                             
            - PyTorch therefore recommends `spawn` for GPU programs.
            - Example error:
                  ```text
                  RuntimeError:
                  Cannot re-initialize CUDA in forked subprocess
                  ```
            - Using `mp.set_start_method("spawn")` avoids this.
      - 2. fork (Linux Default) `mp.set_start_method("fork")`. Child process is created by copying parent. with `fork` Child receives copied memory
            ```table
            Parent Process
                 |
                 ▼
            Child Process
            (copy of parent memory)
            ```
            - Advantages:
                  * Fast
                  * Low overhead
            - Disadvantages:
                  * Unsafe with CUDA
                  * Unsafe with some threads
      3. forkserver `mp.set_start_method("forkserver")`. A dedicated server process creates new child processes. Safer than fork. Less common in PyTorch.            
            ```table
            Main Process
            │
            ▼
            Fork Server
            │
            ▼
            Child Processes
            ```
- force=True → overwrite previously set method. Why force = True?

      - the `force=True` argument forces Python to set the multiprocessing start method to `"spawn"` even if a start method has already been configured earlier in the program. For example, if some code or library previously executed `mp.set_start_method("fork")`, then calling `mp.set_start_method("spawn")` again would raise a `RuntimeError`. By using `force=True`, Python overrides the existing setting and changes it to `"spawn"`, ensuring that each child process `p` created by `mp.Process(...)` starts with a fresh Python interpreter, which is the recommended and safe approach when CUDA devices such as `device = torch.device("cuda:0")` are used.
      
      - Suppose:
            ```python
            mp.set_start_method("spawn")
            ```
            - was already called somewhere.
            - Calling it again:
                  ```python
                  mp.set_start_method("spawn")
                  ```
            - causes:
                  ```text
                  RuntimeError:
                  context has already been set
                  ```
            - Using:
                  ```python
                  mp.set_start_method("spawn", force=True)
                  ```
                  - forces Python to reset it.


#### Q. fork is less recommended than spawn for CUDA workloads. Why?

In PyTorch multiprocessing, `fork` is less recommended than `spawn` for CUDA workloads because `fork` duplicates the parent process without safely reinitializing all its runtime states. Although `fork` initially uses copy-on-write memory, the child inherits the parent's CUDA context state, threads, locks, and other runtime resources. CUDA is not fork-safe after initialization, which can lead to CUDA initialization errors, deadlocks, or unpredictable behavior. `spawn`, on the other hand, starts a fresh Python interpreter, creating an independent process with its own memory space and CUDA initialization. It has higher startup and memory overhead, but provides safer and more predictable CUDA multiprocessing.

**Important:** `fork` does not immediately copy all physical memory; it initially shares memory pages using copy-on-write. `spawn` does not automatically copy the entire parent memory either; objects passed to the child are serialized and transferred. For PyTorch CUDA multiprocessing, prefer `spawn` or `forkserver` rather than `fork`.

#### Q. So CUDA fork concept same as linux fork?

Yes. **PyTorch `fork` uses the same underlying Linux `fork()` system call.** It creates a child process by duplicating the parent's process state using copy-on-write memory.

The difference is how CUDA interacts with the forked process. Linux `fork()` itself works normally, but **CUDA runtime and driver state are not safely inherited by the child process** after CUDA initialization. This can cause errors such as *Cannot re-initialize CUDA in forked subprocess*, deadlocks, or undefined behavior.

**In short:** Linux `fork` is an OS-level process creation mechanism; CUDA's limitation is that its initialized runtime state cannot safely be reused by the forked child. Use `spawn` or `forkserver` for CUDA multiprocessing in PyTorch.


### Single-GPU example

In [ ]:
import torch
import torch.multiprocessing as mp

def gpu_worker(rank):

    device = torch.device("cuda:0")

    x = torch.randn(2000, 2000, device=device)
    y = torch.randn(2000, 2000, device=device)

    z = torch.matmul(x, y)

    torch.cuda.synchronize(device)

    print(f"Worker {rank}: {z.mean().item()}")

if __name__ == "__main__":

    mp.set_start_method("spawn", force=True)

    processes = []

    for rank in range(3):

        p = mp.Process(
            target=gpu_worker,
            args=(rank,)
        )

        p.start()
        processes.append(p)

    for p in processes:
        p.join()

    print("All processes completed")

**What happens internally?**

1. Main process creates three worker processes.
2. Each worker initializes its own CUDA usage.
3. Each worker allocates GPU tensors.
4. Each worker launches matrix multiplication.
5. `torch.cuda.synchronize()` waits for that worker's GPU work.
6. `p.join()` waits for the CPU worker process to terminate.

Notice the difference:

* `torch.cuda.synchronize()` waits for GPU execution.
* `p.join()` waits for process termination.

They solve different synchronization problems.

> *Note: This is an illustrative single-GPU example. Multiple processes on one GPU introduce additional CUDA contexts, memory consumption and scheduling overhead. For many training workloads, one process with multiple CUDA streams is more efficient.*


### Multiprocessing across multiple GPUs

If your system has multiple GPUs, a common architecture is one process per GPU. Use `torch.multiprocessing.spawn()`.
- For example:
    - Process 0 → GPU 0
    - Process 1 → GPU 1
    - Process 2 → GPU 2

`torch.multiprocessing.spawn()` is a PyTorch utility that automatically creates and manages multiple child processes using the `spawn` start method. Instead of manually writing `mp.Process(...)`, `p.start()`, and `p.join()` for each process, `mp.spawn()` launches `nprocs` worker processes, passes a unique process index (`rank`) to each worker function, waits for all processes to finish, and properly handles errors. It is commonly used in `Distributed Data Parallel (DDP)` training where each process controls one GPU. For example:

```python
import torch.multiprocessing as mp

def worker(rank):
    print(f"Process Rank = {rank}")

if __name__ == "__main__":
    mp.spawn(
        fn=worker,
        args=(),
        nprocs=3,
        join=True
    )
```

Execution:

```text
Main Process
 │
 ├── worker(rank=0)
 ├── worker(rank=1)
 └── worker(rank=2)
```

is equivalent to manually creating:

```python
for rank in range(3):
    p = mp.Process(target=worker, args=(rank,))
    p.start()

for p in processes:
    p.join()
```

but `mp.spawn()` is shorter, safer for CUDA, automatically uses the `spawn` method, propagates exceptions from child processes back to the main process, and is therefore the preferred approach for multi-GPU PyTorch applications.


In [ ]:
import torch

gpu_count = torch.cuda.device_count()

print(f"Total GPUs: {gpu_count}")

for i in range(gpu_count):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
import torch
import torch.multiprocessing as mp

def gpu_worker(rank):

    torch.cuda.set_device(rank)

    device = torch.device(f"cuda:{rank}")

    x = torch.randn(2000, 2000, device=device)
    y = torch.randn(2000, 2000, device=device)

    z = torch.matmul(x, y)

    torch.cuda.synchronize(device)

    print(f"GPU {rank}: {z.mean().item()}")

if __name__ == "__main__":

    mp.spawn(
        gpu_worker,
        nprocs=2,
        join=True
    )

## Synchronize multiple processes?

This is an important part of multiprocessing.

In traditional C/C++, you might use:

* Mutex
* Semaphore
* Condition variables
* Thread barriers
* Locks

In Python PyTorch multiprocessing, you can use:

| Synchronization mechanism | Purpose |
| :--- | :--- |
| `Process.join()` | Wait for a process to finish |
| `Event` | Signal another process |
| `Lock` | Protect shared resources |
| `Semaphore` | Control access to limited resources |
| `Barrier` | Wait until all participating processes arrive |
| `Queue` | Exchange data between processes |

Let's understand these with examples.


## Synchronize with join:

`join()` is a process synchronization method provided by Python's `multiprocessing` module. When a parent process creates a child process using `worker_process = Process(target=train_model)` and starts it with `worker_process.start()`, both processes run independently. Calling `worker_process.join()` makes the parent process wait until `worker_process` has completely finished execution. This ensures that all tasks performed inside the child process, including CPU computations, CUDA kernel launches, memory cleanup, and process termination, are completed before the parent process continues.

In PyTorch multiprocessing, `join()` is often used to synchronize worker processes after distributed training or parallel data processing. For example, after creating `rank0_process`, `rank1_process`, and `rank2_process`, the parent process can call `rank0_process.join()`, `rank1_process.join()`, and `rank2_process.join()` to wait for every worker to finish. Unlike `torch.cuda.synchronize()`, which waits only for GPU operations, `join()` synchronizes the entire child process lifecycle. Therefore, `join()` is a **process-level synchronization mechanism**, ensuring that the parent process does not proceed until the specified child process has fully completed and exited.

So `join()` is a **process-level synchronization**, not a **stream-level** or **kernel-level** synchronization.

### Summary

| API | Synchronizes What? |
| :--- | :--- |
| `torch.cuda.synchronize()` | GPU kernels in current process |
| `stream.synchronize()` | One CUDA stream |
| `event.synchronize()` | One CUDA event |
| `Process.join()` | Entire child process |
| `dist.barrier()` | All distributed processes/ranks |

**Important:** `Process.join()` does not tell CUDA to synchronize. It waits for the child process to terminate. Since the child cannot truly finish until its CUDA work is completed and resources are cleaned up, it acts as an indirect synchronization mechanism in multiprocessing programs.


In [1]:
import torch
from multiprocessing import Process

def worker():
    x = torch.randn(10000, 10000, device="cuda")

    y = x @ x    # asynchronous kernel launch

    print("Worker finished Python code")

p = Process(target=worker)

p.start()

print("Waiting...")
p.join()

print("Child process completely finished")

Waiting...
Worker finished Python code
Child process completely finished


## Synchronization using Event

Process 1 performs data preparation. Process 2 must wait until the data is ready before executing GPU computation.

```mermaid
flowchart TD
    subgraph P1 ["Process 1: Data preparation"]
        direction TB
        prep["Prepare input data"] --> set(["event.set()"])
    end

    subgraph P2 ["Process 2: GPU execution"]
        direction TB
        wait(["event.wait()"]) --> exec["Execute GPU computation after signal"]
    end

    set -- "Signal received" --> wait

    style P1 fill:#eef2ff,stroke:#c7d2fe,color:#000000
    style P2 fill:#ecfdf5,stroke:#a7f3d0,color:#000000
    style set fill:#d1fae5,stroke:#6ee7b7,color:#065f46
    style wait fill:#ffedd5,stroke:#fdba74,color:#9a3412
    style prep fill:none,stroke:none,color:#374151
    style exec fill:none,stroke:none,color:#374151
```
* `event.set()` signals that a condition has been satisfied.
* `event.wait()` blocks the CPU process until the event is set.

This synchronizes CPU processes, not CUDA streams.


In [ ]:
import torch
import torch.multiprocessing as mp

def producer(event):
    print("Preparing data...")

    # Data preparation completed
    event.set()

def consumer(event):
    print("Waiting for producer...")

    event.wait()

    print("Starting GPU computation")

    x = torch.randn(1000, 1000, device="cuda")
    y = x @ x

    torch.cuda.synchronize()

    print("GPU computation completed")

if __name__ == "__main__":

    mp.set_start_method("spawn", force=True)

    event = mp.Event()

    p1 = mp.Process(target=producer, args=(event,))
    p2 = mp.Process(target=consumer, args=(event,))

    p1.start()
    p2.start()

    p1.join()
    p2.join()

## Synchronization using Lock mechanism in PyTorch CUDA

#### threading.Lock() [`CPU-side thread synchronization`]:

- In PyTorch CUDA, **locks are used to protect shared resources when multiple CPU threads or processes access the same data or perform operations concurrently**. They prevent race conditions, where multiple threads modify shared variables simultaneously and produce incorrect results. PyTorch provides `threading.Lock()` for CPU-side thread synchronization. For example, when multiple CPU threads access a shared tensor or update a shared counter before launching CUDA operations, a lock ensures that only one thread executes the critical section at a time. Note that `threading.Lock()` does not directly synchronize GPU kernels.
- **Important APIs:** `threading.Lock()` protects CPU shared resources, `torch.cuda.synchronize()` waits for previously submitted GPU work to complete, and `torch.cuda.Event()` helps coordinate and measure GPU execution. For CUDA stream dependencies, use `torch.cuda.Stream.wait_event()` or `torch.cuda.Stream.wait_stream()`. Remember: a lock protects shared access; CUDA synchronization ensures GPU execution dependencies are respected.


In [2]:
import threading
import torch

lock = threading.Lock()
shared_counter = 0

def worker():
    global shared_counter

    with lock:
        shared_counter += 1
        x = torch.randn(1000, device="cuda")
        y = x * 2

        torch.cuda.synchronize()  # Wait for GPU operations

threads = [threading.Thread(target=worker) for _ in range(4)]

for t in threads:
    t.start()

for t in threads:
    t.join()

#### torch.multiprocessing.Lock() [`synchronize multiple processes`]:

- `torch.multiprocessing.Lock()` is used to synchronize **multiple processes** that share resources, such as shared CPU memory, counters, or tensors. It ensures that only one process can access a critical section at a time, preventing race conditions when multiple processes modify the same data. It works similarly to Python's `multiprocessing.Lock()`, but is integrated with PyTorch multiprocessing. You can create it using `torch.multiprocessing.Lock()` and share it with child processes.
- **Difference between `threading.Lock()` and `torch.multiprocessing.Lock()`:** `threading.Lock()` synchronizes multiple threads within the same process, where threads share the same process memory. In contrast, `torch.multiprocessing.Lock()` synchronizes separate processes, which have independent memory spaces but can explicitly share tensors and other resources. For CUDA multiprocessing, PyTorch generally recommends the `spawn` start method rather than `fork`.

    | API | Purpose |
    | :--- | :--- |
    | `threading.Lock()` | Synchronize threads |
    | `torch.multiprocessing.Lock()` | Synchronize processes |
    | `tensor.share_memory_()` | Share CPU tensor memory between processes |
    | `mp.Process()` | Create a separate process |
    | `mp.set_start_method("spawn")` | Start a fresh Python process |
    | `torch.cuda.synchronize()` | Wait for GPU operations to complete |

- **Important CUDA distinction:** `torch.multiprocessing.Lock()` does not synchronize CUDA kernel execution. It protects CPU-side critical sections shared between processes. For GPU execution dependencies, use CUDA streams, events, and `torch.cuda.synchronize()` as appropriate.


####  torch.multiprocessing Code Flow
- code found here PyTorchCUDAandGPU/MultiProcessCode.py

```mermaid
flowchart TD
    subgraph Main ["Main Process"]
        direction TB
        Start(["Start (__main__)"]) --> Init["set_start_method('spawn', force=True)<br/>shared_counter = torch.zeros(1).share_memory_()<br/>lock = mp.Lock()"]
        Init --> Spawn["Create & Start 4 Worker Processes<br/>p.start()"]
        Spawn --> Join["Wait for all processes<br/>p.join()"]
        Join --> Final(["Print 'Final Counter: 4'"])
    end

    subgraph Workers ["4 Spawned Worker Processes (Concurrent)"]
        direction TB
        WStart(["worker(shared_counter, lock)"]) --> Acquire{"with lock:<br/>(Acquire Mutex)"}
        Acquire -- "One process at a time" --> Update["shared_counter[0] += 1<br/>print('Counter:', ...)"]
        Update --> Release["Release lock"]
        Release --> WEnd(["Worker terminates"])
    end

    Spawn -. "Spawns 4 processes" .-> WStart
    WEnd -. "Signals completion" .-> Join
```

## Share memory between process & thread:

##### Q. Tell me how the share memory working in multiple process and thread. One method I can take Global variable.

Yes. You can use a **Global variable** to share data between multiple threads, but global variables alone cannot share memory between independent processes.

The important difference is how threads and processes manage memory.

**1. Sharing memory between threads (`threading.Lock`)**

Multiple threads belong to the same process, so they share the same memory space, including global variables, objects, and tensors. Therefore, you can declare a global variable and allow multiple threads to access or modify it. However, simultaneous modifications can cause race conditions, so use `threading.Lock()` to protect the shared variable.

```python
import threading

shared_counter = 0
lock = threading.Lock()

def worker():
    global shared_counter

    with lock:
        shared_counter += 1
        print(shared_counter)

threads = [
    threading.Thread(target=worker)
    for _ in range(4)
]

for t in threads:
    t.start()

for t in threads:
    t.join()

print("Final:", shared_counter)
```
Here, all four threads access the same shared_counter because they belong to the same process.

**2. Sharing memory between processes (`torch.multiprocessing.Lock`)**

Multiple processes have independent memory spaces. A global variable declared in the parent process is not automatically shared with child processes. With `spawn`, each child starts a fresh Python interpreter. With `fork`, the child initially inherits a copy-on-write view of the parent's memory. To share actual data, use shared memory APIs such as `tensor.share_memory_()` or `multiprocessing.Value()`.

```python
def worker(shared_counter, lock):

    with lock:
        shared_counter[0] += 1
        print("Counter:", shared_counter[0].item())

if __name__ == "__main__":

    mp.set_start_method("spawn")

    # Allocate shared CPU memory
    shared_counter = torch.zeros(1).share_memory_()

    lock = mp.Lock()

    processes = [
        mp.Process(
            target=worker,
            args=(shared_counter, lock)
        )
        for _ in range(4)
    ]

    for p in processes:
        p.start()

    for p in processes:
        p.join()

    print("Final:", shared_counter.item())
```
Here, `shared_counter` uses shared CPU memory, so all four processes can access and modify the same tensor. `mp.Lock()` prevents simultaneous modifications.


## `tensor.share_memory_()` CPU tensor storage accessible across processes:

`tensor.share_memory_()` is a PyTorch API used to move a CPU tensor's storage into shared memory, allowing multiple processes to access the same underlying tensor data without creating separate copies.

Normally, when you create a tensor in a parent process and use `spawn` to create child processes, each process has independent memory. However, `tensor.share_memory_()` makes the tensor's CPU storage shareable between processes. If Process 1 modifies `shared_tensor[0]`, Process 2 can observe that modification because both processes access the same underlying memory. The underscore (`_`) means this operation modifies the tensor's storage in place.

**1. Example without shared memory**

```python
import torch
import torch.multiprocessing as mp

def worker(tensor):
    tensor[0] = 100
    print("Child:", tensor)

if __name__ == "__main__":
    mp.set_start_method("spawn")

    tensor = torch.zeros(3)

    p = mp.Process(target=worker, args=(tensor,))
    p.start()
    p.join()

    print("Parent:", tensor)
```

Expected behavior:

```text
Child: tensor([100., 0., 0.])
Parent: tensor([0., 0., 0.])
```

With `spawn`, the child receives its own tensor storage in this example. Its modification does not change the parent's tensor.

**2. Example with shared memory**

```python
import torch
import torch.multiprocessing as mp

def worker(tensor):
    tensor[0] = 100
    print("Child:", tensor)

if __name__ == "__main__":
    mp.set_start_method("spawn")

    tensor = torch.zeros(3)

    # Enable shared CPU memory
    tensor.share_memory_()

    p = mp.Process(target=worker, args=(tensor,))
    p.start()
    p.join()

    print("Parent:", tensor)
```

Expected output:

```text
Child: tensor([100., 0., 0.])
Parent: tensor([100., 0., 0.])
```

**Shared memory working**

Both processes access the same underlying CPU tensor storage.

## `multiprocessing.Value()`

`multiprocessing.Value()` is a Python multiprocessing API used to create a shared variable that can be accessed and modified by multiple processes. Unlike a normal global variable, `multiprocessing.Value()` allocates the variable in shared memory, allowing parent and child processes to access the same value.

For example, `mp.Value('i', 0)` creates a shared integer initialized to `0`. Here, `'i'` represents a C-style integer type. You can also use `'d'` for a double and `'f'` for a float. By default, `Value()` provides a synchronized wrapper that protects individual read/write operations using an internal lock.

**1. Example: Using `multiprocessing.Value()`**

```python
if __name__ == "__main__":

    mp.set_start_method("spawn")

    # Shared integer variable
    shared_counter = mp.Value('i', 0)

    lock = mp.Lock()

    processes = [
        mp.Process(target=worker, args=(shared_counter, lock))
        for _ in range(4)
    ]

    for p in processes:
        p.start()

    for p in processes:
        p.join()

    print("Final Counter:", shared_counter.value)
```

Expected output:

```text
Counter: 1
Counter: 2
Counter: 3
Counter: 4
Final Counter: 4
```
Here:

* `mp.Value('i', 0)` creates a shared integer.
* `shared_counter.value` reads or modifies its value.
* `mp.Lock()` protects the critical section.
* All four processes access the same shared variable.

> **Note:** `Value()` already has an internal lock by default. The example uses an additional external lock for explicit synchronization. For this example, you could instead use `with shared_counter.get_lock():` and omit `mp.Lock()`.

**2. Difference: `multiprocessing.Value()` vs `tensor.share_memory_()`**

Both APIs share memory between processes, but they serve different purposes.

| Feature | `multiprocessing.Value()` | `tensor.share_memory_()` |
| :--- | :--- | :--- |
| **Purpose** | Share individual variables | Share tensor storage |
| **Data type** | Integer, float, character, etc. | PyTorch tensor data types |
| **Data size** | Single scalar value | Multiple tensor elements |
| **Memory** | Shared memory | Shared CPU memory |
| **Access** | `shared_counter.value` | `shared_tensor[index]` |
| **Synchronization** | Built-in lock by default | No automatic locking |
| **PyTorch integration** | Via `torch.multiprocessing` | Native PyTorch API |
| **Best use** | Counters, flags, status | Datasets, arrays, model parameters |

**3. When should you use each API?**

Use `multiprocessing.Value()` when:

* Sharing a training progress counter.
* Sharing a process status flag.
* Maintaining a shared integer or floating-point value.
* Tracking the number of completed batches.

Use `tensor.share_memory_()` when:

* Sharing large datasets between processes.
* Sharing input tensors between data-processing workers.
* Sharing CPU model parameters.
* Avoiding unnecessary tensor copies between processes.

**4. Important concept for CUDA/PyTorch**

Suppose you have four CPU processes loading training data for your GPU.

```mermaid
flowchart TD
    Shared["<b>Shared CPU memory</b><br/><code>tensor.share_memory_()</code><br/>Common input dataset"]
    P1["<b>Process 1</b><br/>Load batch 1"]
    P2["<b>Process 2</b><br/>Load batch 2"]
    P3["<b>Process 3</b><br/>Load batch 3"]
    P4["<b>Process 4</b><br/>Load batch 4"]
    GPU["<b>NVIDIA GPU</b><br/>Training and model computation"]

    Shared --> P1
    Shared --> P2
    Shared --> P3
    Shared --> P4

    P1 --> GPU
    P2 --> GPU
    P3 --> GPU
    P4 --> GPU

    style Shared fill:#eef2ff,stroke:#c7d2fe,color:#000000
    style GPU fill:#ecfdf5,stroke:#a7f3d0,color:#000000
    style P1 fill:#f9fafb,stroke:#e5e7eb,color:#000000
    style P2 fill:#f9fafb,stroke:#e5e7eb,color:#000000
    style P3 fill:#f9fafb,stroke:#e5e7eb,color:#000000
    style P4 fill:#f9fafb,stroke:#e5e7eb,color:#000000
```

You can use `multiprocessing.Value()` separately to maintain a shared counter that tracks completed batches.

**Interview takeaway:** `multiprocessing.Value()` shares a single scalar variable between processes, whereas `tensor.share_memory_()` shares the underlying storage of an entire CPU tensor. Use `Value()` for counters and flags, and `share_memory_()` for tensor data.


## Atomic operations in PyTorch

**Atomic operations in PyTorch** are operations that execute as a single, indivisible operation. When multiple CPU threads or GPU threads attempt to update the same memory location simultaneously, atomic operations ensure that updates are not lost due to race conditions.

For example, suppose multiple CUDA threads execute `counter += 1` on the same variable. Without atomic protection, two threads might read the same value and overwrite each other's updates. An atomic addition ensures each increment is applied correctly. In PyTorch, `tensor.index_add_()`, `scatter_add_()` and `index_put_(..., accumulate=True)` provide useful accumulation operations, but their GPU implementations are not necessarily deterministic. PyTorch does not expose a general-purpose `atomicAdd()` Python API for arbitrary CUDA tensor operations; custom CUDA kernels can use CUDA atomic functions directly.

**Important APIs**

| API | Purpose |
| :--- | :--- |
| `tensor.index_add_()` | Accumulate values using indices |
| `torch.scatter_add()` | Accumulate values at indexed locations |
| `tensor.scatter_add_()` | In-place indexed accumulation |
| `tensor.index_put_()` | Update tensor using indices |
| `torch.cuda.synchronize()` | Wait for GPU operations to finish |

**Atomic operations vs Locks**

| Feature | Atomic operations | Locks |
| :--- | :--- | :--- |
| **Purpose** | Protect individual memory updates | Protect critical sections |
| **Execution** | Indivisible operation | Lock/unlock mechanism |
| **Granularity** | Usually individual memory locations | Larger code sections |
| **CUDA usage** | GPU threads | Primarily CPU-side synchronization in standard PyTorch |
| **Performance** | Usually lower synchronization overhead | Can introduce contention and waiting |

**Interview takeaway:** Atomic operations prevent race conditions during concurrent memory updates. Locks protect larger critical sections. In CUDA programming, atomic operations are particularly useful when thousands of GPU threads update shared counters, histograms, or gradient accumulation buffers.
